# Day 5: NER, QA, and Summarization

the curriculum says build NER, QA, or summarization, pick one. went with NER as the main thing since our headlines are packed with entities and it's the most useful of the three in real work. also added a small QA demo and a small summarization demo in the same notebook since they don't need separate files, plus a BIO tagging demo since that concept got skipped otherwise.

heads up before starting, a couple of the normal pipeline shortcuts don't exist in this version of transformers anymore. more on that below when we hit them.

In [1]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM

df = pd.read_csv("news_dataset.csv")

print("Loaded", len(df), "headlines")
print("Sample:", df["Title"].iloc[0])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 278 headlines
Sample: Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors


In [2]:
##note: older tutorials use grouped_entities=True here, but that arg was removed in newer transformers versions.
##aggregation_strategy="simple" is the replacement -- it merges word pieces back into whole entities ("Boston Scientific" as one, not "Boston" + "Scientific")
ner = pipeline("ner", aggregation_strategy="simple")
print("NER pipeline loaded.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16281.69it/s]


[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER pipeline loaded.


first fix of the day. that grouped_entities arg used to be the standard way to write this a couple versions back, but it's gone now and throws an error. aggregation_strategy does the same job.

In [3]:
##Run NER over a handful of headlines and see the results
sample_headlines = df["Title"].head(8).tolist()

for headline in sample_headlines:
    entities = ner(headline) ##returns a list of dicts, one per entity found
    print(f"\n{headline}")
    for ent in entities:
        print(f" {ent['word']:25s} -> {ent['entity_group']:6s} (score {ent['score']:.2f})")


Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors
 Elon Musk                 -> PER    (score 0.99)
 SpaceX In                 -> ORG    (score 0.93)

Lisa Su Says AMD's Server Revenue Will Grow More Than 80% This Half -- and That's Not the AI Accelerator Business
 Lisa Su                   -> PER    (score 1.00)
 AMD                       -> ORG    (score 1.00)
 AI                        -> MISC   (score 0.69)

Where Will the Vanguard S&P 500 ETF Be in 20 Years? History Has Good and Bad News for Investors.
 Vanguard                  -> ORG    (score 0.51)
 S                         -> MISC   (score 0.81)
 & P                       -> ORG    (score 0.67)
 500                       -> MISC   (score 0.98)

Stock Market Today, Aug. 18: CoreWeave Falls as Debt-Financing Concerns and Capital Spending Pressures Rise With Interest Rates
 Core                      -> ORG    (score 0.77)
 ##ve                      -> ORG    (score 0.69)

Before You Buy an AI Stock, Co


Mega Cap Growth vs Small Cap Growth: Which ETF Wins?
 ET                        -> ORG    (score 0.66)


mixed results here, which is worth being honest about instead of just showing the good ones.

clean wins: Elon Musk -> PER (0.99), Lisa Su -> PER (1.00), AMD -> ORG (1.00), Nvidia -> ORG (1.00). full confidence, correct entity, boundaries right.

confused ones: SpaceX In grabbed the start of "Investors" and glued it on. S&P 500 got split into S -> MISC and & P -> ORG instead of one thing, the ampersand threw it off. CoreWeave came back as two separate pieces, Core and ##ve, which the merging was supposed to fix but didn't fully. ETF got chopped down to just ET with lower confidence (0.66), the model itself was less sure there.

the reason is this specific model was trained on old 1990s Reuters news (CoNLL-2003). it's never seen ETF, S&P 500, or CoreWeave as training examples, so it's guessing on unfamiliar financial jargon and modern ticker style names. same lesson as Day 4, pretrained-and-frozen only gets you so far on domain specific data, a model fine-tuned on financial text would probably handle this a lot better.

In [4]:
##QA Demo
##the plain "question-answering" pipeline shortcut got removed in this transformers version, so building it directly
##instead using the model class that's still there. this is actually what the shortcut used to do internally anyway.
qa_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
qa_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")

def answer_question(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt")  ##question and context get tokenized together, separated internally by [SEP]
    with torch.no_grad():
        outputs = qa_model(**inputs)
    ##the model doesn't generate text, it predicts WHERE the answer starts and ends in the context
    ##start_logits/end_logits are one score per token position, argmax picks the most likely start and end position
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer_tokens = inputs["input_ids"][0][start:end]  ##slice out just the token span between those two positions
    return qa_tokenizer.decode(answer_tokens)  ##turn those tokens back into readable text

context = "Salesforce shares jumped 8% after the company reported quarterly revenue of 9.3 billion dollars, beating analyst expectations. CEO Marc Benioff credited strong demand for the company's AI products."
question = "How much did Salesforce shares rise?"

answer = answer_question(question, context)
print("Question:", question)
print("Answer:", answer)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 10169.46it/s]

Question: How much did Salesforce shares rise?
Answer: 8 %


second fix of the day, same issue as before. "question-answering" also got removed as a pipeline shortcut in this version, only table and document QA are left. built it manually instead using AutoModelForQuestionAnswering, which still exists.

honestly this ended up being a better way to actually see what's going on anyway. the model doesn't write an answer, it just predicts where the answer starts and ends inside the context, then we slice that exact span out and decode it back to text. that's the whole idea of extractive QA. it got "8 %" straight out of the paragraph, word for word.

no tutorial actually gave me this trick, had to look at what AutoModelForQuestionAnswering + start_logits/end_logits meant and figure out the argmax slicing part myself once the shortcut broke.

In [5]:
##Summarization demo
##"summarization" pipeline shortcut is also gone in this version, same fix as QA -- build it directly
sum_tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6")

long_text = (
    "Salesforce reported quarterly revenue of 9.3 billion dollars on Tuesday, "
    "beating Wall Street expectations by a wide margin. The company said strong "
    "demand for its new AI powered tools drove much of the growth. Shares of the "
    "company jumped 8 percent in after hours trading following the announcement. "
    "CEO Marc Benioff told investors on a call that the company expects even "
    "stronger growth next quarter as more customers adopt its AI products."
)

inputs = sum_tokenizer(long_text, return_tensors="pt", truncation=True)
summary_ids = sum_model.generate(**inputs, max_length=40, min_length=10)  ##generate() produces new tokens one at a time, this isn't extraction
summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\nOriginal:", long_text)
print("\nSummary:", summary)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 358/358 [00:00<00:00, 61051.47it/s]


Original: Salesforce reported quarterly revenue of 9.3 billion dollars on Tuesday, beating Wall Street expectations by a wide margin. The company said strong demand for its new AI powered tools drove much of the growth. Shares of the company jumped 8 percent in after hours trading following the announcement. CEO Marc Benioff told investors on a call that the company expects even stronger growth next quarter as more customers adopt its AI products.

Summary:  Salesforce reported quarterly revenue of 9.3 billion dollars on Tuesday . The company said strong demand for its new AI powered tools drove much of the growth . Shares of the company jumped 8


third and last shortcut that's gone, same story, built it with AutoModelForSeq2SeqLM instead. used a made up longer paragraph here since our actual headlines are already one liners, there's nothing to summarize down further.

this one is a real contrast to the QA step. QA extracted an exact span word for word out of the context. this model uses generate() which produces brand new tokens one at a time, it's not copying text out, it's writing its own version. the output ended up mostly trimming down the original sentences rather than wildly rephrasing, which is a pretty normal thing for BART based summarizers to do, but it's still generated text, not extracted text, which is the actual point (abstractive vs extractive).

In [6]:
##BIO Tagging Demo --same NER model w/o the aggregation that hides the raw tags
raw_ner = pipeline("ner") ##no aggregation_strategy this time so we can see individual tags

sample = "Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors"
raw_tags = raw_ner(sample)

print("\nRaw BIO tags for:", sample)
for tag in raw_tags:
    print(f" {tag['word']:15s} -> {tag['entity']}")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17270.33it/s]


[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Raw BIO tags for: Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors
 El              -> I-PER
 ##on            -> I-PER
 Mu              -> I-PER
 ##sk            -> I-PER
 Space           -> I-ORG
 ##X             -> I-ORG
 In              -> I-ORG


the earlier NER step used aggregation_strategy which actually hides the raw BIO tags, it merges everything into clean entity groups automatically. dropping that shows the real tags underneath.

textbook version of BIO is B marks the start of an entity, I marks a continuation, O means not part of anything. what actually came back was every single tag as I-PER or I-ORG, no B- tag anywhere at all, not even on the very first piece of "Elon" or "SpaceX".

turns out that's not a bug, it's a real quirk of this specific model. the original CoNLL-2003 scheme only really needs a B- tag to tell apart two back to back entities of the same type with nothing between them, which barely happens. so this model learned it can just use I- for basically everything, since a run of I-PER tokens right after something that isn't part of an entity already implies a new entity started there. the textbook explanation is the clean version, what's actually deployed cuts a corner that still works fine almost all the time.

## Takeaway

T5 and BART are both worth mentioning even though we only really touched BART directly (distilbart-cnn-12-6 is a BART variant). both are built around the same idea, text-to-text generation, where every task, translation, summarization, question answering, classification, gets framed as "text goes in, text comes out" instead of needing a different architecture for each job. that's the whole point of an encoder-decoder model like these, one setup handles a bunch of different jobs depending on how you phrase the input.

three of the pipeline shortcuts this project has relied on since day 4, question-answering, summarization, and grouped_entities for NER, are all gone in this version of transformers. had to rebuild each one using the underlying model class directly instead of the convenience wrapper. kind of annoying in the moment but it ended up being a better way to actually understand what each task is doing mechanically, since the shortcut usually hides exactly that.

biggest actual finding of the day is probably the NER results on our own headlines. it's great on generic names and companies but falls apart on financial specific stuff like ETF, S&P 500, and newer tickers like CoreWeave, since the model was trained on old general news, not finance. same story as Day 4, pretrained models are strong in general but need fine-tuning to really work well on a specific domain.